# Session 3 · Dates

**Goal:** turn three columns of text in nine formats into real dates, choose the right century for two-digit years, and catch dates that parse fine but can't be true.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = 3

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
from datetime import date
import pandas as pd
from lnl import check, dates, pipeline
from lnl.check import ___

df = pipeline.clean_through(2)      # everything from sessions 1-2, applied in one line
df[["alias", "date_of_birth", "sentence_date", "projected_release_date"]].head(10)

## 1. What formats are we dealing with?
**Profiling by shape:** replace every digit with `9` and every letter with `A`, then count. It's the fastest way to see what a text column really contains.

In [ ]:
dates.profile_shapes(df["date_of_birth"])

Nine formats mixed in the same column. The shapes like `9/9/99` and `99/99/99` are **two-digit years**, and they're the dangerous ones.

## 2. Why not let pandas guess?
`pd.to_datetime(..., format="mixed")` will try to work out each format on its own. Here's what it does with five real values from this file:

In [ ]:
tricky = pd.Series(["2/19/64", "12/31/99", "5/5/12", "1213-03-22", "14-MAR-2017"],
                   index=["Batman's birth date", "Hulk's release date", "Thor's birth date",
                          "Wonder Woman's birth date", "a sentence date"])
pd.DataFrame({"text": tricky, "pandas guessed": pd.to_datetime(tricky, format="mixed", errors="coerce").astype(str)})

- Batman was **born in 2064**.
- The Hulk's release lands in **1999**, years before he was sentenced.
- Thor was **born in 2012**.
- Wonder Woman's date either **disappears** (pandas 2) or works (pandas 3), depending on which version you happen to have.

Every one of those *looks* like a valid date. That's the problem with guessing. We'll use explicit formats and explicit rules instead.

## 3. Parse with explicit formats

In [ ]:
pd.DataFrame(dates.DATE_FORMATS, columns=["format", "shape (regex)", "how to read it"])

In [ ]:
dob, dob_format = dates.parse_dates(df["date_of_birth"])
display(dob_format.value_counts().to_frame("values"))
print("Present but didn't parse:", (df["date_of_birth"].notna() & dob.isna()).sum())

🧪 **Your turn #1.** How many *sentence* dates use a two-digit year?

In [ ]:
sentence, sentence_format = dates.parse_dates(df["sentence_date"])
answer_1 = ___      # count sentence_format values equal to "M/D/YY"
check.s3_ex1(answer_1)

## 4. Two-digit years: choose the century on purpose
Python's rule for two-digit years is 00-68 → 2000s and 69-99 → 1900s. Spark's rule is different (always 2000s), and Excel's is different again. **None of them know what the column means. We do:**

- A birth date or sentence date **can't be in the future**, so choose the century that isn't.
- A release **can't come before its sentence**, so choose the century that puts it after.

🧪 **Your turn #2.** Before fixing anything, how many parsed birth dates are in the future?

In [ ]:
answer_2 = ___      # compare dob to date.today() with >, then .sum()
check.s3_ex2(answer_2)

In [ ]:
fixed_dob = dates.fix_two_digit_years(dob, dob_format, rule="past")
changed = (dob != fixed_dob).fillna(False).astype(bool)
print(f"Century corrected on {changed.sum():,} birth dates")
pd.DataFrame({"text": df.loc[changed, "date_of_birth"], "parsed": dob[changed], "fixed": fixed_dob[changed]}).head(8)

## 5. Sanity checks: readable is not the same as right
Parsing proves a date is *readable*, not that it's *true*. `dates.session3()` runs all three columns with the century rules, calculates age at sentencing, and flags anything impossible:

In [ ]:
df3 = dates.session3(df)
df3["dob_flag"].value_counts()

In [ ]:
df3.loc[df3["dob_flag"].notna(), ["alias", "date_of_birth", "sentence_date", "age_at_sentencing", "dob_flag"]]

- **Wonder Woman** (born 1213) and **Wolverine** (1882): before 1900.
- **Spider-Man 2099** (born 2071): in the future.
- **Thor**: his ancient birth date was typed as `5/5/12`, which reads as 2012, so he'd have been 13 at sentencing.
- **Winter Soldier** shows up twice: once born 1917 (107 at sentencing, which for him is actually right) and once as `3/10/17`, which the century rule reads as 2017. No rule based on one row can sort that out.

The TDCJ number down the left side is what goes on the review list: it identifies exactly which record a person needs to check. These go to a person for review and are **left out of averages** like age at sentencing. Don't delete them, and don't "fix" them by guessing.

The 1213 date only survives because we store dates as a true date type (`date32`). pandas' default timestamp type can only hold years 1677 to 2262.

## 6. Check across records
A person's birth date shouldn't change between incarcerations. Each row can look fine on its own and still disagree with the same person's other records.

In [ ]:
known = df3.dropna(subset=["sid_number", "date_of_birth"])
births_per_person = known.groupby("sid_number")["date_of_birth"].nunique()
conflicts = births_per_person[births_per_person > 1].index
print(f"{len(conflicts)} people have more than one birth date on file")
df3.loc[df3["sid_number"].isin(conflicts), ["sid_number", "alias", "date_of_birth", "sentence_date"]].sort_values(["sid_number", "sentence_date"]).head(10)

In real corrections data this is common: people give different birth dates at booking. It's a records question, not a coding question. Flag it and route it to the people who can verify; never quietly pick one.

## 7. The date report

In [ ]:
dates.date_report(df, df3)

🧪 **Your turn #3.** How many projected releases fall in 2027?

In [ ]:
answer_3 = ___      # df3["projected_release_date"].dt.year gives the year; compare to 2027 and .sum()
check.s3_ex3(answer_3)

## Trust Test Q4, finally

In [ ]:
print("Earliest projected release:", df3["projected_release_date"].min())
print("Latest projected release:  ", df3["projected_release_date"].max())
df3.to_parquet(PATHS["dir"] / "session3_clean.parquet")     # dates stay dates; the TDCJ number stays the index

The earliest is a real date now, from an earlier incarceration. The latest, 2099-12-31, is the Hulk, placed correctly by the "after the sentence" rule.

## Wrap-up
- **Profile shapes** before parsing anything.
- Use **explicit formats**. Guessing is slow, version-dependent, and silently wrong.
- Two-digit years need a **rule based on what the column means**.
- **Readable ≠ true.** Flag impossible dates for review; don't delete or guess.
- Check **across records** too: one person, one birth date.

**Next session:** integers, money, times, yes/no values, and categories. Then we rerun the Trust Test.

---
## Answer key
```python
answer_1 = (sentence_format == "M/D/YY").sum()
answer_2 = (dob > date.today()).sum()
answer_3 = (df3["projected_release_date"].dt.year == 2027).sum()
```